# XSMB — Phân tích thống kê khám phá

Notebook ví dụ trên dataset `data/xsmb.sqlite`. Mục đích: khảo sát phân bố, tần suất, không phải dự đoán.

**Cài đặt:** `uv sync --extra notebook`

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DB = Path("../data/xsmb.sqlite")
conn = sqlite3.connect(DB)
draws = pd.read_sql("SELECT * FROM draws ORDER BY date", conn, parse_dates=["date"])
numbers = pd.read_sql("SELECT * FROM numbers", conn, parse_dates=["date"])
print(f"{len(draws):,} draws, {draws['date'].min().date()} → {draws['date'].max().date()}")
print(f"{len(numbers):,} numbers (long format)")

## 1. Tần suất 2 số cuối Đặc biệt (toàn lịch sử)

Phân bố lý thuyết: uniform 1/100 = 1%. Bias đáng kể (>2σ) sẽ gợi ý lỗi data, không phải pattern thực.

In [ ]:
db_last2 = numbers.query("prize == 'special'")["last2"].value_counts().sort_index()
expected = len(draws) / 100
print(f"Mean: {db_last2.mean():.1f}, expected: {expected:.1f}")
print(f"Min: {db_last2.min()} ({db_last2.idxmin()}), Max: {db_last2.max()} ({db_last2.idxmax()})")

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(db_last2.index, db_last2.values)
ax.axhline(expected, color="red", linestyle="--", label=f"expected={expected:.0f}")
ax.set_xlabel("2 số cuối Đặc biệt")
ax.set_ylabel("Tần suất")
ax.legend()
ax.set_xticks(range(0, 100, 10))
plt.tight_layout()

## 2. Lô gan — số lâu chưa về

In [ ]:
all_last2 = numbers.groupby("last2")["date"].max().sort_values()
today = draws["date"].max()
gan = (today - all_last2).dt.days.sort_values(ascending=False).head(20)
gan.to_frame("days_since_last")

## 3. Tần suất theo thứ trong tuần

In [ ]:
draws["dow"] = draws["date"].dt.day_name()
draws["db_last2"] = draws["special"].str[-2:]
by_dow = draws.groupby("dow").size().reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
)
by_dow.to_frame("draws")

## 4. Số xuất hiện nhiều nhất (toàn bộ 27 dãy/kỳ, 2 số cuối)

In [ ]:
top = numbers["last2"].value_counts().head(20)
top.to_frame("count")

## 5. Streak — chuỗi ngày liên tiếp 1 số không xuất hiện

Insight: với 27 cơ hội/ngày trên 100 số, kỳ vọng mỗi số xuất hiện ~27% mỗi ngày. Streak "không xuất hiện" ≥ 10 ngày là hiếm nhưng vẫn xảy ra theo phân bố hình học.

In [ ]:
from itertools import groupby

appeared = numbers.groupby(["date", "last2"]).size().unstack(fill_value=0).gt(0)
max_streaks = {}
for n in appeared.columns:
    runs = [sum(1 for _ in g) for k, g in groupby(appeared[n].values) if not k]
    max_streaks[n] = max(runs) if runs else 0
pd.Series(max_streaks).sort_values(ascending=False).head(20).to_frame("max_absent_days")

---

**Lưu ý thống kê:** kết quả xổ số là biến ngẫu nhiên độc lập. Bias quan sát được ở cỡ mẫu N ngày vẫn thấp hơn nhiễu thống kê — không có "số may mắn" thực. Notebook này phục vụ giảng dạy / khám phá dữ liệu, không phải dự đoán.